In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mediapy as mp

from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import load_offline_dataset
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5


## Configuration

Set your local paths and inference parameters here.

In [ ]:
# ===== Paths =====
CLIP_DIR = Path("/path/to/your/offline_clip")
MODEL_PATH = Path("/path/to/your/local/model")
PROCESSOR_PATH = Path("/path/to/your/qwen_processor")
COSMOS_PROCESSOR_PATH = Path("/path/to/your/cosmos_processor")

# ===== Inference config =====
DEVICE = "cuda"
T0_SOD = 175490.23
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98

NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256

IMAGE_SIZE = (448, 800)

os.environ["ALPAMAYO_VLM_PROCESSOR_PATH"] = str(COSMOS_PROCESSOR_PATH)

print("CLIP_DIR =", CLIP_DIR)
print("MODEL_PATH =", MODEL_PATH)
print("PROCESSOR_PATH =", PROCESSOR_PATH)
print("ALPAMAYO_VLM_PROCESSOR_PATH =", os.environ["ALPAMAYO_VLM_PROCESSOR_PATH"])
print("T0_SOD =", T0_SOD)


## Helper functions

In [ ]:
def wrap_to_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def global_motion_yaw_deg(xyz):
    xy = xyz[:, :2]
    disp = xy[-1] - xy[0]
    if np.linalg.norm(disp) < 1e-6:
        return np.nan
    return float(np.rad2deg(np.arctan2(disp[1], disp[0])))


def mean_speed_mps(xyz, dt):
    xy = xyz[:, :2]
    dxy = xy[1:] - xy[:-1]
    step_dist = np.linalg.norm(dxy, axis=1)
    if len(step_dist) == 0:
        return 0.0
    return float(step_dist.mean() / dt)


def segment_mean_speed_table(gt_xyz, model_xyz, dt, segment_sec=1.0):
    seg_len = int(round(segment_sec / dt))
    rows = []
    start = 0
    seg_id = 0

    while start < len(gt_xyz) - 1:
        end = min(start + seg_len + 1, len(gt_xyz))

        def _mean_speed(xyz):
            seg = xyz[start:end]
            if len(seg) < 2:
                return np.nan
            dxy = seg[1:, :2] - seg[:-1, :2]
            return float(np.linalg.norm(dxy, axis=1).mean() / dt)

        rows.append({
            "segment_id": seg_id,
            "t_start_sec": round(start * dt, 2),
            "t_end_sec": round((end - 1) * dt, 2),
            "gt_mean_speed_mps": _mean_speed(gt_xyz),
            "model_mean_speed_mps": _mean_speed(model_xyz),
        })

        start += seg_len
        seg_id += 1

    return pd.DataFrame(rows)


def wrap_to_180_deg(x):
    return (x + 180.0) % 360.0 - 180.0


def yaw_from_rot_plus_x_deg(rot_mats):
    return np.rad2deg(np.arctan2(rot_mats[:, 1, 0], rot_mats[:, 0, 0]))


def history_consistency_table(hist_xyz, hist_rot, dt):
    dxy = hist_xyz[1:, :2] - hist_xyz[:-1, :2]
    step_speed = np.linalg.norm(dxy, axis=1) / dt
    dxy_yaw_deg = np.rad2deg(np.arctan2(dxy[:, 1], dxy[:, 0]))
    rot_yaw_deg = yaw_from_rot_plus_x_deg(hist_rot)[1:]
    yaw_minus_dxy_deg = wrap_to_180_deg(rot_yaw_deg - dxy_yaw_deg)

    return pd.DataFrame({
        "step_idx": np.arange(len(dxy)),
        "dx": dxy[:, 0],
        "dy": dxy[:, 1],
        "step_speed_mps": step_speed,
        "dxy_yaw_deg": dxy_yaw_deg,
        "rot_yaw_deg": rot_yaw_deg,
        "yaw_minus_dxy_deg": yaw_minus_dxy_deg,
        "abs_yaw_minus_dxy_deg": np.abs(yaw_minus_dxy_deg),
    })


def plot_trajectory_compare(hist_xyz, gt_xyz, pred_xyz, title):
    plt.figure(figsize=(8, 8))
    plt.plot(hist_xyz[:, 0], hist_xyz[:, 1], "o-", linewidth=2, markersize=3, label="History")
    plt.plot(gt_xyz[:, 0], gt_xyz[:, 1], "k-o", linewidth=3, markersize=4, label="GT")
    plt.plot(pred_xyz[:, 0], pred_xyz[:, 1], "o-", linewidth=2, markersize=3, label="Pred")
    plt.scatter([0.0], [0.0], c="red", marker="x", s=140, label="t0 ego")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Load model and processor

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map="cuda:0",
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print("Model and processor loaded.")


## Load offline local data

In [ ]:
data = load_offline_dataset(
    clip_dir=CLIP_DIR,
    t0_sod=T0_SOD,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    debug=True,
    image_size=IMAGE_SIZE,
)

print("Offline dataset loaded.")
print("camera_indices:", data["camera_indices"].tolist())
print("image_frames shape:", tuple(data["image_frames"].shape))
print("ego_history_xyz shape:", tuple(data["ego_history_xyz"].shape))
print("ego_history_rot shape:", tuple(data["ego_history_rot"].shape))
print("ego_future_xyz shape:", tuple(data["ego_future_xyz"].shape))
print("ego_future_rot shape:", tuple(data["ego_future_rot"].shape))
print("requested frame indices:", data["video_frame_indices"][0].tolist())
print("actual frame indices (first camera):", data["actual_video_frame_indices"][0].tolist())


## Show sampled images

In [ ]:
mp.show_images(
    data["image_frames"].flatten(0, 1).permute(0, 2, 3, 1).cpu().numpy(),
    columns=4,
    width=220,
)


## History motion / yaw consistency diagnosis

In [ ]:
hist_xyz = data["ego_history_xyz"].cpu().numpy()[0, 0, :, :]
hist_rot = data["ego_history_rot"].cpu().numpy()[0, 0, :, :, :]

df_hist_consistency = history_consistency_table(hist_xyz, hist_rot, TIME_STEP)

print("[History consistency summary]")
display(pd.DataFrame([{
    "mean_abs_yaw_minus_dxy_deg": float(df_hist_consistency["abs_yaw_minus_dxy_deg"].mean()),
    "median_abs_yaw_minus_dxy_deg": float(df_hist_consistency["abs_yaw_minus_dxy_deg"].median()),
    "last3_mean_abs_yaw_minus_dxy_deg": float(df_hist_consistency["abs_yaw_minus_dxy_deg"].tail(3).mean()),
    "last5_mean_abs_yaw_minus_dxy_deg": float(df_hist_consistency["abs_yaw_minus_dxy_deg"].tail(5).mean()),
}]))

print("\n[History consistency tail]")
display(df_hist_consistency.tail(5))


## Build model inputs

In [ ]:
messages = helper.create_message(
    frames=data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)

print("seq length:", inputs.input_ids.shape)

model_inputs = {
    "tokenized_data": inputs,
    "ego_history_xyz": data["ego_history_xyz"],
    "ego_history_rot": data["ego_history_rot"],
}
model_inputs = helper.to_device(model_inputs, DEVICE)


## Run inference

In [ ]:
if DEVICE.startswith("cuda"):
    torch.cuda.manual_seed_all(42)
    autocast_context = torch.autocast("cuda", dtype=torch.bfloat16)
else:
    autocast_context = torch.autocast(device_type=DEVICE, enabled=False)

with autocast_context:
    pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
        data=model_inputs,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        return_extra=True,
    )

print("Inference done.")


## Main metrics and diagnosis

In [ ]:
cot_value = extra["cot"][0]
if hasattr(cot_value, "tolist"):
    cot_value = cot_value.tolist()

print("Chain-of-Causation:")
print(cot_value)

hist_xyz = data["ego_history_xyz"].cpu().numpy()[0, 0, :, :]
gt_xyz = data["ego_future_xyz"].cpu().numpy()[0, 0, :, :]

pred_xyz_np = pred_xyz.cpu().numpy()[0, 0, :, :, :]
gt_xy = gt_xyz[:, :2].T
pred_xy = pred_xyz_np[:, :, :2].transpose(0, 2, 1)

diff = np.linalg.norm(pred_xy - gt_xy[None, ...], axis=1).mean(-1)
best_idx = int(diff.argmin())
min_ade = float(diff.min())

pred_best_xyz = pred_xyz_np[best_idx]

gt_final_xy = gt_xyz[-1, :2]
pred_final_xy = pred_best_xyz[-1, :2]
final_point_error = float(np.linalg.norm(pred_final_xy - gt_final_xy))

gt_mean_speed = mean_speed_mps(gt_xyz, TIME_STEP)
pred_mean_speed = mean_speed_mps(pred_best_xyz, TIME_STEP)
speed_error = float(pred_mean_speed - gt_mean_speed)

gt_yaw = global_motion_yaw_deg(gt_xyz)
pred_yaw = global_motion_yaw_deg(pred_best_xyz)
if np.isfinite(gt_yaw) and np.isfinite(pred_yaw):
    yaw_error = float(np.rad2deg(wrap_to_pi(np.deg2rad(pred_yaw - gt_yaw))))
else:
    yaw_error = np.nan

df_metrics = pd.DataFrame([{
    "best_sample_idx": best_idx,
    "minADE_m": min_ade,
    "final_point_error_m": final_point_error,
    "gt_final_x": float(gt_final_xy[0]),
    "gt_final_y": float(gt_final_xy[1]),
    "pred_final_x": float(pred_final_xy[0]),
    "pred_final_y": float(pred_final_xy[1]),
    "gt_mean_speed_mps": gt_mean_speed,
    "pred_mean_speed_mps": pred_mean_speed,
    "speed_error_mps": speed_error,
    "gt_yaw_deg": gt_yaw,
    "pred_yaw_deg": pred_yaw,
    "yaw_error_deg": yaw_error,
    "cot": str(cot_value),
}])

print("[Main metrics]")
display(df_metrics)


## Segment speed profile

In [ ]:
df_speed_profile = segment_mean_speed_table(
    gt_xyz=gt_xyz,
    model_xyz=pred_best_xyz,
    dt=TIME_STEP,
    segment_sec=1.0,
)

print("[Segment speed profile]")
display(df_speed_profile)


## Plot best trajectory

In [ ]:
plot_trajectory_compare(
    hist_xyz=hist_xyz,
    gt_xyz=gt_xyz,
    pred_xyz=pred_best_xyz,
    title=f"Offline inference @ t0={T0_SOD:.2f}, minADE={min_ade:.3f}m",
)


## Plot all predicted trajectories

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(pred_xyz_np.shape[0]):
    plt.plot(
        pred_xyz_np[i, :, 0],
        pred_xyz_np[i, :, 1],
        "o-",
        linewidth=1.5,
        markersize=2.5,
        alpha=0.35,
        label=f"Pred #{i+1}" if i < 5 else None,
    )

plt.plot(gt_xyz[:, 0], gt_xyz[:, 1], "k-o", linewidth=3, markersize=4, label="GT")
plt.plot(hist_xyz[:, 0], hist_xyz[:, 1], "o-", linewidth=2, markersize=3, label="History")
plt.scatter([0.0], [0.0], c="red", marker="x", s=140, label="t0 ego")
plt.xlabel("x")
plt.ylabel("y")
plt.title("All predicted trajectories")
plt.grid(True, alpha=0.3)
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.tight_layout()
plt.show()
